# Contrast Task PLSC on Whole-Brain cFos Density

Two analyses, answering different questions:

**Main text (Sections 5-11).** *Contrast task PLSC*, unrotated. `R = C' X` is
built directly from the seven orthogonal design contrasts and the row-normalized
rows are each contrast's brain salience vector. No SVD, so no rotation, so every
contrast keeps its own identity instead of being mixed across latent variables.
The row norm is the multivariate magnitude statistic; a per-region OLS on the
same normalized data supplies per-region inference.

**Supplement (Section 12).** *Mean-centered task PLS*, the conventional variant
used by the BraiAn pipeline (Chiaruttini et al. 2025, Cell Rep 44:115876).

---

## What changed in this version

### 1. Region inclusion is no longer listwise-complete

Listwise completeness cost **89 of 360 regions**, and the main path does not need
it: `R` is built from design-cell means, and a cell mean is defined as soon as
that cell has one observation. Three region sets are now built explicitly, and
every section states which it uses.

| set | rule | used by |
|---|---|---|
| `CORE` | complete across all mice | per-mouse gain centering, the global scalar, split-half |
| `ANALYSIS` | every design cell has >= `MIN_OBS_PER_CELL` | `R`, saliences, per-region inference, BSR, all Figure 5 exports |
| `NULLSET` | `CORE` | every permutation null |

Two constraints drive this split, and both are about permutation rather than
tidiness:

- **A norm is not comparable across region sets.** `||R_j||` grows with the
  number of regions, so the observed magnitude and its null must be computed over
  the *same* set. The permutation nulls therefore run on `NULLSET`, and
  `R_mag_null` (not `R_mag`) is what carries the p-value. `R_mag` on the wider
  `ANALYSIS` set is reported descriptively alongside.
- **A shuffle can empty a cell.** Under a sex shuffle or a within-perfusion-batch
  row permutation, which cell loses a region's missing animals depends on the
  draw, so per-cell `n` becomes permutation-dependent and the null drifts.
  Section 2a computes the worst-case-safe set for each scheme and prints the
  sizes, so the cost of the conservative choice is visible rather than assumed.

### 2. The per-region OLS is now a NaN-aware closed form

A single `lstsq` over all regions requires complete data. The model is saturated
with an orthogonal +/-1 cell basis, so `beta = B8' m / 8` **exactly**, under
unequal *and* missing `n`. Cell means, pooled variance and per-coefficient SEs are
computed directly, giving correct SEs at per-region degrees of freedom. The
`beta == R_cellmean / 8` identity is still asserted and still holds.

Regions with fewer than `MIN_TOTAL_N_FOR_SE` observations have zero or negative
residual df: their saliences are kept, their p-values are `NaN`, and they are
excluded from FDR rather than silently contributing.

### 3. Per-mouse gain centering stays on `CORE`

Centering each mouse on whatever regions it happens to have would confound
brain-wide gain with coverage: a mouse missing ten hypothalamic nuclei would get
a different "whole-brain level" for arithmetic reasons. Gain is therefore computed
over identical anatomy for every mouse, then subtracted from the wider matrix.
Same reason the global scalar is `CORE`-only.

### 4. Fixes

- `residualize_batch` extracted the batch BLUP with
  `dd["batch"].map(m.random_effects).astype(float)`, mapping to length-1 Series.
  Now takes the scalar explicitly, matching the global LMM notebook, and drops
  NaN rows before fitting.
- `group_region_means` double-indexed on its `idx` branch (`m = m[idx]` then
  `mat[idx][m]`). Unused, so it never fired; removed.
- `perm_lv` re-permuted a second time inside the `not HAS_PERF` branch, so the
  index it had just computed was discarded.
- Dead `if False else` expression in the Panel 5E export.

### 5. Sign convention

Unchanged and unchanged deliberately: `cre/+`, `M`, `light` are `+1`. **The global
LMM has now been flipped to match this**, so the note in earlier versions about
one convention needing to be flipped is resolved — all three notebooks are
control-positive. Genotype-bearing numbers from any earlier global-model run are
negated relative to this one.

## The `GAIN_NORMALIZE` switch

The main analysis is *defined* on mean-subtracted data, so `True` is the only
meaningful setting for Sections 5-11. The switch exists for the Section 12
supplement and the Section 4a diagnostic.

## 1. Config and imports

In [ ]:
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.linalg import svd
from scipy.stats import pearsonr, norm, t as tdist
from statsmodels.stats.multitest import multipletests
from statsmodels.tools.sm_exceptions import ConvergenceWarning

rng = np.random.default_rng(42)

# ============================================================================
# PATHS
# ============================================================================
# ---------------------------------------------------------------------------
# Paths -- repo-relative, so this notebook runs wherever the repository is
# checked out. PROJECT_ROOT is found by walking up from the working directory
# until a folder containing `data/` appears. If you keep the data outside the
# repository, set PROJECT_ROOT by hand and everything below follows.
# ---------------------------------------------------------------------------
def find_project_root(markers=("data", "notebooks")):
    """Nearest ancestor directory containing all of `markers`. Requiring both
    `data/` and `notebooks/` means a stray `data/` folder somewhere on the path
    cannot be mistaken for the repository root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if all((candidate / m).is_dir() for m in markers):
            return candidate
    raise RuntimeError(
        f"No ancestor of {here} contains {list(markers)}. Set PROJECT_ROOT by hand."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"

MATRIX_CSV = DATA_DIR / "compiled" / "cleaned_output" / "density_by_region_leaves_wide.csv"
INFO_CSV   = DATA_DIR / "mouse_info_template.csv"

# ### FIXED ### the panel exports used to be written to `figure_data_b` while
# notebook 05 read `figure_data`, so the atlas heatmaps were rendered from a
# stale directory unless it was copied across by hand. One directory now.
FIGURE_DIR      = RESULTS_DIR / "figures";     FIGURE_DIR.mkdir(exist_ok=True, parents=True)
FIGURE_DATA_DIR = RESULTS_DIR / "figure_data"; FIGURE_DATA_DIR.mkdir(exist_ok=True, parents=True)

REGION_ACRONYM_COL   = "Acronym"
MEASUREMENT_COL_FLAG = "annotation_measurements"

# ============================================================================
# ### CHOICE ### R_MODE -- how R = C'X is built
# ============================================================================
#   "cellmean" -> R = C' M, M the 8 x p matrix of design-cell means.
#                 Identical to animal-level weighting by 1/n_cell (asserted in
#                 Section 5). Contrasts exactly orthogonal. Every cell counts
#                 equally, so the n=3 cell is as influential as the n=8 cell:
#                 the correct estimand, a noisier one.
#   "animal"    -> R = C_animal' X, unweighted. Sample-weighted estimand.
#                 Contrasts NOT orthogonal under unequal n -- the correlation
#                 matrix is printed so the departure is visible, not assumed.
R_MODE = "cellmean"

# ============================================================================
# ### CHOICE ### REGION INCLUSION
# ============================================================================
#   "complete"      -> listwise complete across all mice (the old behaviour).
#   "cell_complete" -> a region is analysed if EVERY design cell has at least
#                      MIN_OBS_PER_CELL observations. R is built from cell means,
#                      so this is all the main path actually requires.
#
# Permutation nulls always run on the CORE (listwise-complete) set regardless,
# because a norm is not comparable across region sets and a shuffle can empty a
# cell. Section 2a prints all the counts.
REGION_INCLUSION  = "cell_complete"
MIN_OBS_PER_CELL  = 1

# Residual df for the per-region OLS is (n_obs - 8). Below this many total
# observations a region has no usable SE; its salience is kept and its p-value is
# NaN rather than a number computed from zero or negative df.
MIN_TOTAL_N_FOR_SE = 10

# ### CHOICE ### GAIN_NORMALIZE
# Subtract each mouse's brain-wide mean before analysis.
#   True  -> saliences describe spatial REDISTRIBUTION only. Required for the
#            main path: the whole point is to ask what survives gain removal.
#   False -> saliences contain global gain. Section 12 supplement / diagnostics.
GAIN_NORMALIZE = True

RESIDUALIZE  = True      # remove immuno-batch BLUPs before anything else
ZSCORE_REGIONS = True    # z-score each region across animals after centering

# ============================================================================
# INFERENCE
# ============================================================================
N_PERM   = 10_000        # permutation null for contrast magnitudes
N_BOOT   = 10_000        # bootstrap for BSR  (raise to 100k for final)
N_SPLITS = 5_000         # split-half, Section 12 only
FDR_Q    = 0.20          # discovery threshold for the per-region OLS
FDR_Q_STRICT = 0.05
BSR_THRESH = 2.0
TOPK     = 25
PSEUDO   = None          # set in Section 2 to half the smallest non-zero density

RUN_TAG = f"{R_MODE}_gain{'off' if GAIN_NORMALIZE else 'on'}"

matplotlib.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42,
                            "savefig.bbox": "tight", "savefig.dpi": 300,
                            "font.size": 8, "axes.spines.top": False,
                            "axes.spines.right": False})

GROUP_COLORS = {("cre/+", "M"): "#2E5E8E", ("cre/+", "F"): "#7BA7CC",
                ("cre/cre", "M"): "#B03A2E", ("cre/cre", "F"): "#E08E82"}
GROUP_LABEL  = {("cre/+", "M"): "Control M", ("cre/+", "F"): "Control F",
                ("cre/cre", "M"): "MKO M",    ("cre/cre", "F"): "MKO F"}

print(f"RUN_TAG = {RUN_TAG}")
print(f"  R_MODE          = {R_MODE}")
print(f"  GAIN_NORMALIZE  = {GAIN_NORMALIZE}")
print(f"  N_PERM / N_BOOT = {N_PERM} / {N_BOOT}")

## 2. Load and transform

Three matrices, not interchangeable. Every downstream cell states which it uses.

| matrix | contents | used by |
|---|---|---|
| `Xl` | log density, raw | global scalar, Section 11 regression panels |
| `Xa` | `Xl` with batch BLUPs removed | input to `Xz` only |
| `Xz` | `Xa`, per-mouse centered, region z-scored | `R`, the per-region OLS, Section 12 |

### CHOICE ### Pseudocount is half the smallest non-zero density in the matrix,
computed once here and reused everywhere. Methods must state the value, not the
rule.

In [ ]:
raw = pd.read_csv(MATRIX_CSV, index_col=REGION_ACRONYM_COL)
meas_cols = [c for c in raw.columns if MEASUREMENT_COL_FLAG in c]
X_raw = raw[meas_cols].copy()
X_raw.columns = [c.replace(f"_{MEASUREMENT_COL_FLAG}", "") for c in X_raw.columns]

info = pd.read_csv(INFO_CSV)
info["id"] = info["id"].astype(str)
info = info.set_index("id").loc[X_raw.columns]

HAS_BATCH  = "batch" in info.columns
HAS_PERF   = "perfusion_batch" in info.columns
HAS_LITTER = "litter" in info.columns
for flag, name in [(HAS_BATCH, "batch"), (HAS_PERF, "perfusion_batch"), (HAS_LITTER, "litter")]:
    if not flag:
        print(f"!! no '{name}' column -- dependent sections will be skipped")

# --- design cell index, needed before the inclusion rule can be applied -------
CELLS_DESIGN = list(product(["cre/+", "cre/cre"], ["M", "F"], ["D", "L"]))
cell_of  = pd.Series(list(zip(info.genotype, info.sex, info.light)), index=info.index)
cell_idx = np.array([CELLS_DESIGN.index(c) for c in cell_of])
CELL_POS = [np.where(cell_idx == i)[0] for i in range(len(CELLS_DESIGN))]
n_cell   = np.array([len(p) for p in CELL_POS])
assert (n_cell > 0).all(), "an empty design cell -- the saturated model is not identified"

# --- inclusion masks ---------------------------------------------------------
present   = X_raw.notna().values                      # (n_regions_all, n_mice)
core_mask = present.all(axis=1)

cellwise = np.ones(len(X_raw), dtype=bool)
for c in range(len(CELLS_DESIGN)):
    cellwise &= present[:, cell_idx == c].sum(axis=1) >= MIN_OBS_PER_CELL

analysis_mask = core_mask if REGION_INCLUSION == "complete" else cellwise
if REGION_INCLUSION not in ("complete", "cell_complete"):
    raise ValueError(REGION_INCLUSION)

print(f"regions in matrix               : {len(X_raw)}")
print(f"  listwise complete (CORE)      : {int(core_mask.sum())}")
print(f"  >= {MIN_OBS_PER_CELL}/cell (cell-complete)     : {int(cellwise.sum())}")
print(f"  -> ANALYSIS set ({REGION_INCLUSION}) : {int(analysis_mask.sum())}")
print(f"     gained over listwise       : +{int(analysis_mask.sum() - core_mask.sum())}")

X_raw   = X_raw[analysis_mask]
regions = X_raw.index.tolist()
n_reg   = len(regions)

# Positions of the CORE regions WITHIN the analysis set. Gain centering and the
# global scalar are computed over these columns only, so every mouse is averaged
# over identical anatomy -- otherwise "whole-brain level" is confounded with
# coverage.
core_in_analysis = core_mask[analysis_mask]
CORE_POS = np.flatnonzero(core_in_analysis)
assert len(CORE_POS) > 0, "no listwise-complete regions -- gain centering is undefined"
print(f"     of which CORE             : {len(CORE_POS)}  (used for gain + global scalar)")

PSEUDO = float(np.nanmin(X_raw.values[X_raw.values > 0])) / 2
print(f"\npseudocount = {PSEUDO:.6g}  (half the smallest non-zero density)")

Xl = np.log(X_raw.T.values + PSEUDO)          # n x p, log density; may contain NaN
n_mice, p_reg = Xl.shape
n_missing_cells = int(np.isnan(Xl).sum())
print(f"Xl: {n_mice} mice x {p_reg} regions, {n_missing_cells} missing values "
      f"({100*n_missing_cells/Xl.size:.2f}%)")

# ### CHOICE ### The global scalar is a CORE-only quantity, for the same reason
# gain centering is. This is the same construction as the global-model notebook's
# `global_cfos_raw`, so the two must agree numerically.
info["global_cfos"] = Xl[:, CORE_POS].mean(axis=1)
print(f"\nper-mouse whole-brain mean (over {len(CORE_POS)} CORE regions) spans "
      f"{info.global_cfos.min():.2f} to {info.global_cfos.max():.2f} log units "
      f"({np.exp(info.global_cfos.max() - info.global_cfos.min()):.0f}x in density)")

print("\nDesign cells:")
for i, c in enumerate(CELLS_DESIGN):
    print(f"  {str(c):28s} n={n_cell[i]}")
print(f"  total n = {n_cell.sum()}")
if n_cell.min() < 4:
    j = int(np.argmin(n_cell))
    print(f"  !! smallest cell {CELLS_DESIGN[j]} has n={n_cell[j]} -- disclose in Methods")

## 2a. Which region set each permutation scheme can use

A permutation null is only valid if the set of regions contributing to the
statistic is the **same on every draw**. With missing data that is not automatic:
under a sex shuffle or a within-perfusion-batch row permutation, which design cell
loses a region's missing animals depends on the draw, so per-cell `n` -- and
therefore whether a cell mean exists at all -- becomes permutation-dependent. The
null then drifts, and it drifts in the anticonservative direction.

The cell below computes, for each scheme, a **worst-case-safe** region set: the
regions whose per-cell count cannot fall below `MIN_OBS_PER_CELL` under *any*
permissible relabeling. Within a stratum, a shuffle can only move animals between
that stratum's cells, so the worst case for a cell is its within-stratum size minus
every missing animal in that stratum, summed over strata.

This is printed as a diagnostic. The nulls themselves run on `CORE`, which is
worst-case-safe under every scheme by construction. If the numbers below show that
a worst-case-safe set is much larger than `CORE` for the scheme you care about,
that is the argument for switching `NULLSET` — and the reason it is a visible
number rather than a buried assumption.

In [ ]:
def worst_case_cell_counts(present_mat, cidx, strata_key):
    """Lower bound on each region's per-cell count over all within-stratum permutations.

    present_mat : (n_regions, n_mice) bool
    cidx        : (n_mice,) design-cell index
    strata_key  : (n_mice,) anything hashable; permutation is restricted within these
    """
    n_reg = present_mat.shape[0]
    worst = np.zeros((n_reg, len(CELLS_DESIGN)))
    for s in pd.unique(pd.Series(strata_key)):
        m = np.asarray(strata_key) == s
        n_missing = (~present_mat[:, m]).sum(axis=1)             # (n_reg,)
        for c in range(len(CELLS_DESIGN)):
            n_cs = int(((cidx == c) & m).sum())
            if n_cs:
                worst[:, c] += np.maximum(n_cs - n_missing, 0)
    return worst


present_analysis = ~np.isnan(Xl.T)          # (n_reg, n_mice), analysis set

SCHEMES = {}
if HAS_PERF:
    # row permutation moves whole brain vectors within a perfusion batch
    SCHEMES["row shuffle within perfusion_batch"] = info["perfusion_batch"].values
# unrestricted sex shuffle: one stratum, so the bound is the most conservative
SCHEMES["unrestricted sex shuffle"] = np.zeros(n_mice, dtype=int)
if HAS_LITTER:
    SCHEMES["sex shuffle within litter"] = info["litter"].values

print("WORST-CASE-SAFE REGION SET, BY PERMUTATION SCHEME")
print("=" * 72)
print(f"  CORE (listwise complete)                     {len(CORE_POS):>4} / {n_reg}")
safe_masks = {}
for name, key in SCHEMES.items():
    w = worst_case_cell_counts(present_analysis, cell_idx, key)
    m = w.min(axis=1) >= MIN_OBS_PER_CELL
    safe_masks[name] = m
    print(f"  {name:<44} {int(m.sum()):>4} / {n_reg}")

all_safe = np.logical_and.reduce(list(safe_masks.values()))
print(f"  {'safe under EVERY scheme above':<44} {int(all_safe.sum()):>4} / {n_reg}")

# ### CHOICE ### NULLSET = CORE. It is worst-case-safe under every scheme without
# needing an argument, and the norm-comparability constraint means the observed
# magnitude has to be recomputed on it anyway. Switch to `all_safe` only if the
# count above makes it worth restating the magnitudes.
NULL_POS = CORE_POS
print(f"\nNULLSET = CORE ({len(NULL_POS)} regions). Permutation magnitudes are computed")
print("over this set; the wider ANALYSIS set carries the saliences and per-region")
print("inference. The two magnitudes are NOT comparable and are reported separately.")

## 3. Map each region to its major CCFv3 division

Organizational only. Does not affect any statistic. The twelve top-level
grey-matter divisions are also the set used for the subdivision enrichment
analysis, so the mapping is shared.

In [ ]:
from brainglobe_atlasapi import BrainGlobeAtlas

MAJOR_DIVISIONS = ["Isocortex", "OLF", "HPF", "CTXsp", "STR", "PAL",
                   "TH", "HY", "MB", "P", "MY", "CB"]
DIVISION_ORDER = MAJOR_DIVISIONS

atlas = BrainGlobeAtlas("allen_mouse_25um")
_div_cache = {}

def get_major_division(acronym):
    if acronym in _div_cache:
        return _div_cache[acronym]
    out = "Other/unassigned"
    try:
        st = atlas.structures[acronym]
        for anc_id in reversed(st["structure_id_path"]):
            anc = atlas.structures[anc_id]["acronym"]
            if anc in MAJOR_DIVISIONS:
                out = anc
                break
    except KeyError:
        pass
    _div_cache[acronym] = out
    return out

region_division = np.array([get_major_division(r) for r in regions])
div_counts = pd.Series(region_division).value_counts().reindex(
    DIVISION_ORDER + ["Other/unassigned"]).dropna().astype(int)
print(div_counts.to_string())
assert div_counts.sum() == n_reg, "division counts must sum to n regions"

## 4. Preprocessing: batch residualization, then the gain switch

### CHOICE ### [P1] `R` has no random effects. It will find batch variance and
hand it back, and because a design factor can be entangled with batch structure,
a real effect and a staining axis look identical. Subtract **only** the batch
BLUP; fixed effects are what the analysis is meant to find, so they stay in.

### CHOICE ### [P3] Then subtract each mouse's brain-wide level. A brain that
stains hot is hot in every region, so "all regions move together" is the single
largest source of covariance in the matrix. Removing it is what makes the main
analysis a test of *redistribution* rather than a restatement of the global
result.

### CHOICE ### Region z-scoring is a linear rescale within each region, so it
cannot change any per-region p-value. It makes estimates comparable across
regions, which is what the volcano and the salience join need.

In [ ]:
def residualize_batch(Xl_, info_, region_list):
    """Per-region LMM with immuno batch as a random intercept; subtract only the
    batch-attributable shift (BLUP), leaving fixed effects intact.

    ### FIXED ###
      1. The BLUP was extracted with `.map(m.random_effects).astype(float)`, which
         maps each batch to a length-1 Series rather than to a number. Now takes
         the scalar explicitly, matching the global-model notebook.
      2. NaN rows are dropped before fitting and the BLUP is applied only where the
         region is observed, so a region with missing animals is still corrected
         instead of failing.
    """
    Xr, failed = Xl_.copy(), []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        warnings.simplefilter("ignore", RuntimeWarning)
        for j, r in enumerate(region_list):
            y = Xl_[:, j]
            ok = ~np.isnan(y)
            if ok.sum() < 10:
                failed.append(r)
                continue
            dd = info_.loc[ok].copy()
            dd["y"] = y[ok]
            try:
                m = smf.mixedlm("y ~ genotype * sex * light", dd,
                                groups=dd["batch"]).fit()
                blup_map = {k: float(np.asarray(v).ravel()[0])
                            for k, v in m.random_effects.items()}
                blup = info_["batch"].map(blup_map).astype(float).values
                Xr[:, j] = y - np.where(np.isnan(blup), 0.0, blup)
            except Exception:
                failed.append(r)
    if failed:
        print(f"  !! batch residualization failed for {len(failed)} regions "
              f"(left unadjusted): {failed[:8]}{'...' if len(failed) > 8 else ''}")
    return Xr


if RESIDUALIZE and HAS_BATCH:
    Xa = residualize_batch(Xl, info, regions)
    print(f"batch BLUPs removed from {n_reg} regions")
else:
    Xa = Xl.copy()
    print("!! not residualizing (RESIDUALIZE=False or no batch column)")

Xz = Xa.copy()

# ### CHOICE ### Per-mouse gain is computed over CORE regions ONLY, then
# subtracted from every column. Centering each mouse on whatever regions it
# happens to have would make "whole-brain level" a function of coverage: a mouse
# missing ten hypothalamic nuclei would get a different gain estimate for purely
# arithmetic reasons, and that difference would then be pushed into every region.
if GAIN_NORMALIZE:
    gain_per_mouse = Xz[:, CORE_POS].mean(axis=1, keepdims=True)
    Xz = Xz - gain_per_mouse
    print(f"per-mouse gain removed (estimated over {len(CORE_POS)} CORE regions)")

if ZSCORE_REGIONS:
    mu = np.nanmean(Xz, axis=0, keepdims=True)
    sd = np.nanstd(Xz, axis=0, ddof=1, keepdims=True)
    sd[~np.isfinite(sd) | (sd == 0)] = 1.0
    Xz = (Xz - mu) / sd

info["global_cfos_resid"] = Xa[:, CORE_POS].mean(axis=1)
print(f"Xz built: per-mouse centered = {GAIN_NORMALIZE}, region z-scored = {ZSCORE_REGIONS}")
print(f"  {int(np.isnan(Xz).sum())} missing values remain in Xz "
      f"(handled by NaN-aware cell means throughout)")

### 4a. How much is gain? (diagnostic)

Quantifies what the switch removes: the share of variance in `Xa` carried by the
per-mouse brain-wide mean, versus the region profile and the residual. Also
reports how much of that gain is attributable to design cell versus immuno
batch -- both numbers belong in the manuscript, the second in Limitations.

In [ ]:
# NaN-aware: the decomposition is over observed values only, and the per-mouse
# gain term uses CORE columns so it matches the centering applied in Section 4.
_reg   = np.nanmean(Xa, axis=0, keepdims=True)
_dev   = Xa - _reg
_gain  = _dev[:, CORE_POS].mean(axis=1, keepdims=True)
_resid = _dev - _gain

ss = lambda A: float(np.nansum(A ** 2))
ss_reg  = ss(np.broadcast_to(_reg - np.nanmean(Xa), Xa.shape))
ss_gain = ss(np.broadcast_to(_gain, Xa.shape))
ss_res  = ss(_resid)
tot     = ss_reg + ss_gain + ss_res

print("Variance decomposition of Xa (batch-residualized log density):")
for lab, v in [("region profile", ss_reg), ("per-mouse gain", ss_gain), ("residual", ss_res)]:
    print(f"  {lab:18s} {100*v/tot:5.1f}%")

g = pd.Series(_gain.ravel(), index=info.index)
cell_lab = info.genotype.astype(str) + "_" + info.sex.astype(str) + "_" + info.light.astype(str)
def _r2(y, grp):
    gm = y.groupby(grp).transform("mean")
    return 1 - ((y - gm) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"\ngain variance explained by design cell = {100*_r2(g, cell_lab):.1f}%")
if HAS_BATCH:
    print(f"gain variance explained by immuno batch = {100*_r2(g, info['batch']):.1f}%")
    print("  ^ report this in Limitations: batch and genotype are partly confounded,")
    print("    so no permutation scheme fully separates regional effects from gain.")

## 4b. The global effect, tested directly

One value per mouse, full factorial, no multiplicity problem. This is the
**primary confirmatory statistic**; the per-contrast magnitude in Section 7 is
the multivariate co-gatekeeper. Everything per-region is decomposition.

Fit to `global_cfos` from **`Xl`** -- raw log density, not residualized --
because the model carries batch as a random intercept. Fitting the residualized
mean would remove batch twice.

In [ ]:
assert not np.allclose(info["global_cfos"], info["global_cfos_resid"]) or not RESIDUALIZE, \
    "global_cfos should come from Xl, not Xa"

THREE_WAY_TERM = "genotype[T.cre/cre]:sex[T.M]:light[T.L]"

if HAS_BATCH:
    scalar_fit = smf.mixedlm("global_cfos ~ genotype * sex * light", info,
                             groups=info["batch"]).fit()
else:
    scalar_fit = smf.ols("global_cfos ~ genotype * sex * light", info).fit()
    print("!! no batch column -- OLS without a batch random effect")
print(scalar_fit.summary())

print(f"\nthree-way (natural log)  = {scalar_fit.params[THREE_WAY_TERM]:+.4f}")
print(f"three-way (log2)         = {scalar_fit.params[THREE_WAY_TERM]/np.log(2):+.4f}")
print("  ^ figures report log2; tables must match or state the conversion.")

### 4b-i. Contrasts on the global scalar

All contrasts reconstructed from the eight fitted cell means, so the reference
coding of the fit is irrelevant to any estimate or SE.

`norm.sf` rather than `1 - norm.cdf`: the latter underflows to exactly zero once
`|z|` exceeds about 8.3, turning a real result into an unreportable one.
`log10_p` is carried alongside so small values stay legible.

In [ ]:
fe  = scalar_fit.fe_params if hasattr(scalar_fit, "fe_params") else scalar_fit.params
cov = scalar_fit.cov_params().loc[fe.index, fe.index]

def cell_weight(geno, sex, light):
    g, s, l = int(geno == "cre/cre"), int(sex == "M"), int(light == "L")
    w = pd.Series(0.0, index=fe.index)
    w["Intercept"] = 1
    for key, val in [("genotype[T.cre/cre]", g), ("sex[T.M]", s), ("light[T.L]", l),
                     ("genotype[T.cre/cre]:sex[T.M]", g * s),
                     ("genotype[T.cre/cre]:light[T.L]", g * l),
                     ("sex[T.M]:light[T.L]", s * l),
                     (THREE_WAY_TERM, g * s * l)]:
        if key in w.index:
            w[key] = val
    return w.values

CELLS_DESIGN = list(product(["cre/+", "cre/cre"], ["M", "F"], ["D", "L"]))
cells = {c: cell_weight(*c) for c in CELLS_DESIGN}

def contrast(wvec, label):
    est = float(wvec @ fe)
    se  = float(np.sqrt(wvec @ cov @ wvec))
    z   = est / se
    return {"contrast": label, "estimate_ln": est, "estimate_log2": est / np.log(2),
            "se": se, "z": z, "p": float(2 * norm.sf(abs(z))),
            "log10_p": float(np.log10(2) + norm.logsf(abs(z)) / np.log(10))}

rows = []
for g in ["cre/+", "cre/cre"]:
    for sx in ["M", "F"]:
        rows.append(contrast(cells[(g, sx, "L")] - cells[(g, sx, "D")],
                             f"light | {g} {sx}"))
for sx in ["M", "F"]:
    for l in ["D", "L"]:
        rows.append(contrast(cells[("cre/cre", sx, l)] - cells[("cre/+", sx, l)],
                             f"MKO-control | {sx} {l}"))
for l in ["D", "L"]:
    rows.append(contrast(
        (cells[("cre/cre", "M", l)] - cells[("cre/+", "M", l)]) -
        (cells[("cre/cre", "F", l)] - cells[("cre/+", "F", l)]),
        f"geno:sex | {l}"))
rows.append(contrast(
    ((cells[("cre/cre", "M", "L")] - cells[("cre/cre", "M", "D")]) -
     (cells[("cre/+", "M", "L")] - cells[("cre/+", "M", "D")])) -
    ((cells[("cre/cre", "F", "L")] - cells[("cre/cre", "F", "D")]) -
     (cells[("cre/+", "F", "L")] - cells[("cre/+", "F", "D")])),
    "geno:sex:light"))

global_contrasts = pd.DataFrame(rows)
print(global_contrasts.to_string(index=False,
      formatters={"estimate_ln": "{:+.3f}".format, "estimate_log2": "{:+.3f}".format,
                  "se": "{:.3f}".format, "z": "{:+.2f}".format, "p": "{:.4g}".format,
                  "log10_p": "{:.1f}".format}))
global_contrasts.to_csv(FIGURE_DATA_DIR / f"global_contrasts_{RUN_TAG}.csv", index=False)

## 5. Contrast design matrix

Seven contrasts: three main effects and four interactions. In a fully crossed
2x2x2 these are **mutually orthogonal by construction**, which answers McIntosh
& Lobaugh's caveat that unrotated PLS is hard to interpret with non-orthogonal
contrasts. Orthogonality is verified below rather than asserted.

### CHOICE ### Sign convention: `control`, `M`, and `light` are `+1`. A positive
salience therefore means higher cFos in control, in males, or in the light --
matching the per-region convention. Note this is the *opposite* of the global
LMM's treatment coding (control as reference, so positive genotype = MKO). The
manuscript must state one convention and flip the other; see the printout.

The two `R_MODE` options and their equivalence are established here.

In [ ]:
code_map = {"cre/+": 1, "cre/cre": -1, "M": 1, "F": -1, "D": -1, "L": 1}
NAMES = ["genotype", "sex", "light",
         "geno:sex", "geno:light", "sex:light", "geno:sex:light"]
SEX_TERMS = {"sex", "geno:sex", "sex:light", "geno:sex:light"}
GSL_IDX   = NAMES.index("geno:sex:light")

def contrast_row(g, s, l):
    cg, cs, cl = code_map[g], code_map[s], code_map[l]
    return [cg, cs, cl, cg*cs, cg*cl, cs*cl, cg*cs*cl]

C = np.array([contrast_row(*c) for c in CELLS_DESIGN], float)     # 8 x 7
k_con = C.shape[1]

cell_of  = pd.Series(list(zip(info.genotype, info.sex, info.light)), index=info.index)
cell_idx = np.array([CELLS_DESIGN.index(c) for c in cell_of])
CELL_POS = [np.where(cell_idx == i)[0] for i in range(len(CELLS_DESIGN))]
n_cell   = np.array([len(p) for p in CELL_POS])

print("Design cells:")
for i, c in enumerate(CELLS_DESIGN):
    print(f"  {str(c):28s} n={n_cell[i]}")
print(f"  total n = {n_cell.sum()}")
if n_cell.min() < 4:
    j = int(np.argmin(n_cell))
    print(f"  !! smallest cell {CELLS_DESIGN[j]} has n={n_cell[j]} -- disclose in Methods")

# --- orthogonality of the cell-level contrast matrix -----------------------
G = C.T @ C
assert np.allclose(G, np.diag(np.diag(G))), "cell-level contrasts must be orthogonal"
print(f"\ncell-level C: orthogonal, every column norm = sqrt({int(G[0,0])})")

# --- animal-level design ---------------------------------------------------
C_animal = C[cell_idx]                                            # n x 7
Ga = C_animal.T @ C_animal
off = Ga - np.diag(np.diag(Ga))
print(f"animal-level C (unweighted): max |off-diagonal| = {np.abs(off).max():.1f} "
      f"-> {'orthogonal' if np.abs(off).max() < 1e-9 else 'NOT orthogonal (unequal n)'}")

### 5a. `R = C' X` and the 1/n equivalence

The identity that makes "individual values, `1/n_cell` weighted" and "cell
means" the same analysis:

```
sum_i c_i x_i / n_cell(i)  =  sum_cells c_cell * mean_cell  =  C' M
```

Asserted numerically below. If it ever fails, something upstream has broken the
cell indexing.

In [ ]:
def cell_means(X, cidx):
    """8 x p matrix of design-cell means, NaN-aware.

    ### CHANGED ### `np.nanmean`, not `.mean`. With the relaxed inclusion rule a
    region may be missing individual animals; the cell mean is still defined as
    long as the cell retains one observation, which the ANALYSIS mask guarantees.
    A cell that IS empty for a region yields NaN, which propagates -- that is
    intended, not a bug: it should poison the estimate rather than be silently
    treated as zero.
    """
    with np.errstate(invalid="ignore"):
        return np.vstack([np.nanmean(X[cidx == i], axis=0)
                          for i in range(len(CELLS_DESIGN))])

def build_R(X, cidx, mode=None):
    """R = C'X, k x p. See R_MODE for what the two modes mean."""
    mode = mode or R_MODE
    if mode == "cellmean":
        return C.T @ cell_means(X, cidx)
    elif mode == "animal":
        return C[cidx].T @ X
    raise ValueError(f"unknown R_MODE {mode!r}")

# ### the equivalence, asserted ###
# The 1/n equivalence holds exactly only where nothing is missing, so it is
# asserted on the CORE columns. It is an identity about the WEIGHTING, not about
# the data, so demonstrating it on the complete subset is sufficient.
w_inv_n  = 1.0 / n_cell[cell_idx]
R_weighted = (C[cell_idx] * w_inv_n[:, None]).T @ Xz[:, CORE_POS]
R_cellmean = C.T @ cell_means(Xz[:, CORE_POS], cell_idx)
assert np.allclose(R_weighted, R_cellmean, atol=1e-9), \
    "1/n_cell weighted animal-level R must equal the cell-mean R"
print("OK  1/n_cell-weighted animal-level R == cell-mean R  (max abs diff "
      f"{np.abs(R_weighted - R_cellmean).max():.2e})")
print("    -> these are one analysis, not two. The real choice is weighted vs unweighted.")

R_animal = C[cell_idx].T @ np.nan_to_num(Xz[:, CORE_POS])
cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print("\nrow-wise cosine similarity, cellmean vs unweighted-animal R:")
for j, nm in enumerate(NAMES):
    print(f"  {nm:16s} {cos(R_cellmean[j], R_animal[j]):+.4f}")

### 5b. Row normalization

Each row of `R` is one contrast's covariance profile across regions. Two objects
come out of it:

- **magnitude** `||R_j||` -- the multivariate effect size for contrast *j*, and
  the statistic the permutation test in Section 7 acts on. Non-negative, so its
  null is bounded at zero and right-skewed: **the p-value is one-tailed.**
  Several current figure legends say "two-sided" and are wrong.
- **unit salience** `R_j / ||R_j||` -- the direction, comparable across
  contrasts because every row now has length 1.

### CHOICE ### BSR in Section 8 is computed on the **raw** `R`, not the unit
rows. Normalizing inside each bootstrap replicate would divide out exactly the
magnitude variability the SE is meant to capture.

In [ ]:
R_obs  = build_R(Xz, cell_idx)                       # k x p, ANALYSIS set
R_mag  = np.linalg.norm(R_obs, axis=1)               # descriptive magnitude
R_unit = R_obs / R_mag[:, None]

# ### The magnitude that carries the p-value ###
# A norm grows with the number of regions, so the observed statistic and its null
# MUST be computed over the same set. The permutation null (Section 7) runs on
# NULLSET, so the tested magnitude is this one, not R_mag above.
R_obs_null = build_R(Xz[:, NULL_POS], cell_idx)
R_mag_null = np.linalg.norm(R_obs_null, axis=1)

assert np.isfinite(R_obs).all(), \
    ("R contains non-finite values -- some region has an empty design cell. "
     "Raise MIN_OBS_PER_CELL or set REGION_INCLUSION='complete'.")

mag_df = pd.DataFrame({"contrast": NAMES,
                       "magnitude_analysis": R_mag,
                       "magnitude_nullset": R_mag_null}).sort_values(
    "magnitude_nullset", ascending=False)
print(f"ANALYSIS set: {n_reg} regions      NULLSET: {len(NULL_POS)} regions\n")
print(mag_df.to_string(index=False, formatters={
    "magnitude_analysis": "{:.3f}".format, "magnitude_nullset": "{:.3f}".format}))
print("\n  magnitude_analysis is DESCRIPTIVE only -- it is not the statistic tested.")
print("  magnitude_nullset is what Section 7 supplies a p-value for.")
print("  Do not quote one with the other's p. (p-values in Section 7; magnitude")
print("  alone is not evidence.)")

## 6. Per-region OLS on the normalized data

**This is the significance test for the main path.** Fit per region on `Xz` --
batch-residualized, per-mouse centered, region z-scored -- with the same +/-1
contrast coding, so each coefficient *is* a contrast.

### CHOICE ### OLS, not a mixed model. `Xz` already has batch BLUPs removed;
adding batch as a random intercept would remove it twice. Stated in Methods.

### CHOICE ### The model is saturated (8 parameters, 8 cells), so
`beta = D_cell' M / 8` exactly, even unbalanced -- meaning the coefficients equal
`R_cellmean / 8`. The OLS is therefore not an independent second analysis; it is
the salience with a standard error attached. Asserted below. Report it that way.

Standard errors *do* depend on cell n, which is what the saturated identity does
not give you and why the OLS is worth running.

Vectorized: one `lstsq` for all regions at once. `df = n - 8`.

In [ ]:
# ### CHANGED ### NaN-aware saturated closed form, replacing the single lstsq.
#
# The model is saturated (8 cells, 8 parameters) with an orthogonal +/-1 cell
# basis B8 = [1 | C], so B8' B8 = 8I and therefore
#
#     beta = B8' m / 8            m = the 8 cell means
#
# EXACTLY, under unequal n and under missing data alike. Coefficient k puts weight
# B8[c, k] / 8 on cell c, so
#
#     var(beta_k) = sigma2 * sum_c (B8[c,k] / 8)^2 / n_c
#
# with sigma2 the pooled within-cell variance at (n_obs - 8) df. This is the same
# closed form the per-region permutation notebook uses, and it is exact rather
# than an approximation for the missing-data case.
B8 = np.column_stack([np.ones(len(CELLS_DESIGN)), C])          # 8 x 8
assert np.allclose(B8.T @ B8, 8 * np.eye(8)), "cell basis is not orthogonal"

n_obs_cell = np.vstack([(~np.isnan(Xz[cell_idx == i])).sum(axis=0)
                        for i in range(len(CELLS_DESIGN))]).astype(float)   # 8 x p
M_cell = cell_means(Xz, cell_idx)                                            # 8 x p

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    ssr = np.vstack([np.nansum((Xz[cell_idx == i] - M_cell[i]) ** 2, axis=0)
                     for i in range(len(CELLS_DESIGN))])                     # 8 x p
    n_tot    = n_obs_cell.sum(axis=0)                                        # p
    df_resid = n_tot - len(CELLS_DESIGN)                                     # p
    sigma2   = np.where(df_resid > 0, ssr.sum(axis=0) / np.where(df_resid > 0, df_resid, 1),
                        np.nan)

    beta = (B8.T @ M_cell) / 8.0                                             # 8 x p
    inv_n = np.where(n_obs_cell > 0, 1.0 / np.where(n_obs_cell > 0, n_obs_cell, 1), np.nan)
    var_factor = ((B8 / 8.0) ** 2).T @ inv_n                                 # 8 x p
    se = np.sqrt(sigma2[None, :] * var_factor)

    tstat = beta / se
    pval  = 2 * tdist.sf(np.abs(tstat), np.maximum(df_resid, 1)[None, :])

# Regions without usable residual df keep their salience but get no p-value.
no_se = (n_tot < MIN_TOTAL_N_FOR_SE) | ~np.isfinite(sigma2)
pval[:, no_se] = np.nan
se[:, no_se]   = np.nan
tstat[:, no_se] = np.nan
if no_se.any():
    print(f"!! {int(no_se.sum())} region(s) have < {MIN_TOTAL_N_FOR_SE} observations; "
          f"salience kept, p-value NaN, excluded from FDR:")
    print(f"   {[regions[i] for i in np.flatnonzero(no_se)][:15]}")

print(f"residual df per region: min {int(np.nanmin(df_resid))}, "
      f"median {int(np.nanmedian(df_resid))}, max {int(np.nanmax(df_resid))}")

# --- the saturated identity, asserted --------------------------------------
assert np.allclose(beta[1:], R_obs / 8, atol=1e-8, equal_nan=True), \
    "saturated OLS coefficients must equal R_cellmean / 8"
print(f"OK  OLS coefficients == R / 8  (max abs diff "
      f"{np.nanmax(np.abs(beta[1:] - R_obs/8)):.2e})")
print("    The OLS adds standard errors, not a new estimand.\n")

# --- long table, FDR within contrast ---------------------------------------
rows = []
for j, nm in enumerate(NAMES):
    q = np.full(n_reg, np.nan)
    ok = np.isfinite(pval[j + 1])
    if ok.any():
        q[ok] = multipletests(pval[j + 1][ok], alpha=FDR_Q, method="fdr_bh")[1]
    rows.append(pd.DataFrame({
        "contrast": nm, "region": regions, "division": region_division,
        "estimate": beta[j + 1], "se": se[j + 1], "t": tstat[j + 1],
        "p": pval[j + 1], "q": q, "n_obs": n_tot, "df_resid": df_resid,
        "salience": R_obs[j], "unit_salience": R_unit[j]}))
region_lmm = pd.concat(rows, ignore_index=True)
region_lmm["neglog10_p"] = -np.log10(np.maximum(region_lmm["p"], 1e-300))
region_lmm["sig_q20"] = region_lmm["q"] < FDR_Q
region_lmm["sig_q05"] = region_lmm["q"] < FDR_Q_STRICT

summary = (region_lmm.groupby("contrast")
           .agg(n_q20=("sig_q20", "sum"), n_q05=("sig_q05", "sum"),
                n_tested=("p", lambda x: int(np.isfinite(x).sum())))
           .reindex(NAMES))
summary["magnitude_analysis"] = R_mag
summary["magnitude_nullset"]  = R_mag_null
print("Per-region OLS on mean-subtracted data:")
print(summary.to_string())
print(f"\n^ If these counts collapse relative to the uncentered per-region models,")
print("  that is the result: the effects live in global amplitude, not in")
print("  region-specific redistribution. Non-zero counts are the 'mixture' case.")

### 6a. Baseline-density check

Row-centering removes the per-mouse constant but **not** the slope-below-one
seen in the Section 11 regression. If surviving regions track baseline density,
they are the compression effect reappearing, not spatial redistribution.

Run this before interpreting any survivor as structure.

In [ ]:
baseline = np.nanmean(Xl, axis=0)                # per-region mean log density (raw)
base_rows = []
for j, nm in enumerate(NAMES):
    _fin = np.isfinite(baseline) & np.isfinite(beta[j + 1])
    r, p_ = pearsonr(baseline[_fin], np.abs(beta[j + 1][_fin]))
    sig = region_lmm[(region_lmm.contrast == nm) & region_lmm.sig_q20]
    base_rows.append({"contrast": nm,
                      "r_baseline_vs_abs_estimate": r, "p": p_,
                      "n_sig_q20": len(sig),
                      "mean_baseline_sig": baseline[[regions.index(x) for x in sig.region]].mean()
                                            if len(sig) else np.nan,
                      "mean_baseline_all": np.nanmean(baseline)})
baseline_check = pd.DataFrame(base_rows)
print(baseline_check.to_string(index=False))
print("\nIf |r| is large, residualize on baseline before calling survivors 'structure'.")
baseline_check.to_csv(FIGURE_DATA_DIR / f"baseline_check_{RUN_TAG}.csv", index=False)

## 7. Permutation nulls for the contrast magnitudes

Two schemes, and they are different *kinds* of test. Their p-values are not
comparable in magnitude and Methods must say so.

| contrast family | scheme | what it is |
|---|---|---|
| `light`, `genotype`, `geno:light` | permute rows within perfusion batch | design-based randomization test |
| anything with `sex` | shuffle sex labels | population-level test of association |

### FLAG ### Sex was never assigned by anyone. The sex-involving nulls cannot
fully null an interaction, so those p-values are **reported as untested** in the
manuscript rather than as significance. This includes the three-way.

### CHOICE ### A litter-blocked version of the three-way null is computed
alongside, since litters are split by sex at weaning and mice are therefore not
independent draws under an unrestricted sex shuffle.

**One-tailed**, because the magnitude is a norm.

In [ ]:
def perm_rows_within_perfusion(info, rng):
    out = np.arange(len(info))
    for b in info["perfusion_batch"].unique():
        m = np.where(info["perfusion_batch"].values == b)[0]
        out[m] = rng.permutation(m)
    return out

def _cellidx_from_sex(info, new_sex):
    return np.array([CELLS_DESIGN.index(c)
                     for c in zip(info.genotype, new_sex, info.light)])

def _safe_R(X, cidx):
    """None if any design cell emptied out -- a shuffle can do that.

    Also rejects a draw that produced a non-finite R. On NULLSET (complete data)
    that cannot happen, but the guard means the function stays correct if NULL_POS
    is ever pointed at a wider set."""
    if len(np.unique(cidx)) < len(CELLS_DESIGN):
        return None
    R_ = build_R(X, cidx)
    return None if not np.isfinite(R_).all() else R_

# ### CHANGED ### every null runs on NULLSET (see Section 2a): a norm is not
# comparable across region sets, and a shuffle can make per-cell n
# permutation-dependent where data are missing.
Xz_null = Xz[:, NULL_POS]


def run_perm(n_perm=N_PERM, rng=rng):
    null_design = np.full((n_perm, k_con), np.nan)
    null_sex    = np.full((n_perm, k_con), np.nan)
    null_litter = np.full(n_perm, np.nan)
    sex_vals = info["sex"].values

    for i in range(n_perm):
        if HAS_PERF:
            idx = perm_rows_within_perfusion(info, rng)
            Rp = _safe_R(Xz_null[idx], cell_idx)
            if Rp is not None:
                null_design[i] = np.linalg.norm(Rp, axis=1)

        Rp = _safe_R(Xz_null, _cellidx_from_sex(info, rng.permutation(sex_vals)))
        if Rp is not None:
            null_sex[i] = np.linalg.norm(Rp, axis=1)

        if HAS_LITTER:
            new_sex = sex_vals.copy()
            for lt in info["litter"].unique():
                m = np.where(info["litter"].values == lt)[0]
                new_sex[m] = rng.permutation(sex_vals[m])
            Rp = _safe_R(Xz_null, _cellidx_from_sex(info, new_sex))
            if Rp is not None:
                null_litter[i] = np.linalg.norm(Rp, axis=1)[GSL_IDX]
    return null_design, null_sex, null_litter

null_design, null_sex, null_litter = run_perm()

def one_tailed_p(null, obs):
    null = null[~np.isnan(null)]
    return (np.sum(null >= obs) + 1) / (len(null) + 1), len(null)

rows = []
for j, nm in enumerate(NAMES):
    if nm in SEX_TERMS:
        nl, scheme = null_sex[:, j], "sex_shuffle (population-level, UNTESTED for interactions)"
    else:
        nl, scheme = null_design[:, j], "within_perfusion_batch (design-based)"
    p_, n_ok = one_tailed_p(nl, R_mag_null[j])
    mc_se = np.sqrt(p_ * (1 - p_) / max(n_ok, 1))
    rows.append({"contrast": nm, "magnitude": R_mag_null[j],
                 "magnitude_analysis_set": R_mag[j], "p_one_tailed": p_,
                 "mc_se": mc_se, "n_valid_perm": n_ok, "floor": 1 / (n_ok + 1),
                 "at_floor": p_ <= 1.5 / (n_ok + 1), "scheme": scheme})
contrast_results = pd.DataFrame(rows)

if HAS_LITTER:
    p_lit, n_lit = one_tailed_p(null_litter, R_mag_null[GSL_IDX])
    contrast_results["p_litter"] = np.where(
        contrast_results.contrast == "geno:sex:light", p_lit, np.nan)
else:
    contrast_results["p_litter"] = np.nan

print(contrast_results.to_string(index=False,
      formatters={"magnitude": "{:.3f}".format, "p_one_tailed": "{:.4f}".format,
                  "mc_se": "{:.4f}".format, "floor": "{:.5f}".format}))
print("\nAny row with at_floor=True must be reported as p < floor, never as p = 0.")
print(f"`magnitude` is over NULLSET ({len(NULL_POS)} regions) and is the quantity the")
print(f"p-value refers to. `magnitude_analysis_set` ({n_reg} regions) is descriptive.")
contrast_results.to_csv(FIGURE_DATA_DIR / f"contrast_magnitudes_{RUN_TAG}.csv", index=False)

### 7a. Does the row permutation hold sex fixed?

`perm_rows_within_perfusion` moves a mouse's whole brain vector onto another
mouse's label triple. If perfusion batches are mixed-sex, that shuffles sex too,
and the "design-based" label on the non-sex contrasts is reasoning from a false
premise. Settle it empirically.

In [ ]:
if HAS_PERF:
    ct = pd.crosstab(info["perfusion_batch"], info["sex"])
    print(ct.to_string())
    mixed = ((ct > 0).sum(axis=1) > 1).any()
    print(f"\n-> perfusion batches are {'MIXED-sex' if mixed else 'single-sex'}.")
    if mixed:
        print("   The row permutation DOES shuffle sex. Describe the non-sex nulls as")
        print("   a joint within-batch null in Methods, not as genotype/light-only.")
    else:
        print("   Sex is held fixed; the design-based label is correct.")

## 8. Bootstrap stability (BSR)

BSR = mean salience across resamples / bootstrap SE, per region per contrast.

### FLAG ### BSR is a **stability** measure, not a significance test, and it is
**uncorrected across regions**. Manuscript language must be "stable
contributors", never "reliably shift their activity".

### FIXED ### The usual normal-theory chance rate for `|BSR| > 2` is 4.6%.
**In this design it is not.** Simulation on null data with these exact cell
sizes (5,4,6,5,8,5,4,3) gives an empirical rate of **~9%**, roughly double, and
the inflation is stable across `n_boot` so it is not a convergence artifact. It
is small-n bootstrap SE bias: resampling within cells of 3-8 animals produces
cell means less variable than the true sampling distribution, so the SE is too
small and BSR is inflated.

Reporting the normal-theory 4.6% would therefore understate the chance count by
about half. This cell computes an **empirical** BSR null by permutation instead,
and that is the number the manuscript should quote.

No Procrustes rotation: `C` is fixed by construction, so unlike an SVD there is
no sign or order ambiguity across replicates. Resampling is stratified within
design cell.

In [ ]:
def bootstrap_R(X, n_boot, rng):
    """Streaming mean/SE of R over bootstrap resamples. No large allocation.

    ### CHANGED ### NaN-aware, with a PER-REGION valid-replicate count. A resample
    within a cell of 3-8 animals can draw only the animals that are missing for a
    given region, emptying that cell and making R undefined there. Accumulating
    with `np.nansum` and dividing by a per-region count keeps every other region's
    SE correct instead of poisoning the whole column.
    """
    p_ = X.shape[1]
    s1 = np.zeros((k_con, p_)); s2 = np.zeros((k_con, p_))
    cnt = np.zeros((k_con, p_))
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, size=len(pos), replace=True)
                              for pos in CELL_POS])
        Rb = build_R(X[idx], cell_idx[idx])
        good = np.isfinite(Rb)
        s1 += np.where(good, Rb, 0.0)
        s2 += np.where(good, Rb ** 2, 0.0)
        cnt += good
    with np.errstate(invalid="ignore", divide="ignore"):
        m   = np.where(cnt > 0, s1 / np.maximum(cnt, 1), np.nan)
        var = np.maximum(s2 / np.maximum(cnt, 1) - m ** 2, 0) * cnt / np.maximum(cnt - 1, 1)
    boot_ok = cnt >= 0.9 * n_boot
    if not boot_ok.all():
        print(f"  !! {int((~boot_ok).sum())} of {boot_ok.size} contrast-region cells had "
              f"< 90% valid bootstrap replicates (a thin cell's resample can drop every "
              f"observed animal); their BSR is less trustworthy.")
    return m, np.sqrt(var)

R_boot_mean, R_boot_se = bootstrap_R(Xz, N_BOOT, rng)
BSR = np.divide(R_boot_mean, R_boot_se, out=np.zeros_like(R_boot_mean),
                where=R_boot_se > 0)

# ### FIXED ### Empirical BSR null. Break the design by permuting animal labels,
# rerun the bootstrap, and record the |BSR| > threshold rate. This absorbs the
# small-n SE bias that the normal-theory rate ignores.
N_BSR_NULL   = 20          # permutation replicates; each runs a full bootstrap
N_BOOT_NULL  = max(500, N_BOOT // 20)   # cheaper inner loop; the rate is stable in n_boot

null_rates = []
for _ in range(N_BSR_NULL):
    perm = rng.permutation(n_mice)
    mb, sb = bootstrap_R(Xz[perm], N_BOOT_NULL, rng)
    Bn = np.divide(mb, sb, out=np.zeros_like(mb), where=sb > 0)
    null_rates.append(float(np.nanmean(np.abs(Bn) > BSR_THRESH)))

emp_rate     = float(np.mean(null_rates))
emp_rate_sd  = float(np.std(null_rates))
exp_chance   = emp_rate * n_reg                       # per contrast
exp_normal   = 2 * norm.sf(BSR_THRESH) * n_reg

print(f"Empirical |BSR| > {BSR_THRESH} rate on permuted data: "
      f"{emp_rate:.4f} (sd {emp_rate_sd:.4f}, {N_BSR_NULL} permutations)")
print(f"  normal-theory rate would be {2*norm.sf(BSR_THRESH):.4f} "
      f"-> inflation factor {emp_rate / (2*norm.sf(BSR_THRESH)):.2f}x")
print(f"  expected stable regions per contrast by chance: {exp_chance:.0f} of {n_reg}")
print(f"  (normal theory would have said {exp_normal:.0f} -- do not quote that number)\n")

print(f"Observed regions with |BSR| > {BSR_THRESH}:")
for j, nm in enumerate(NAMES):
    obs = int(np.nansum(np.abs(BSR[j]) > BSR_THRESH))
    print(f"  {nm:16s} {obs:4d}   ({'above' if obs > exp_chance else 'at or below'} chance)")

pd.DataFrame({"threshold": BSR_THRESH, "empirical_rate": emp_rate,
              "empirical_rate_sd": emp_rate_sd, "normal_theory_rate": 2*norm.sf(BSR_THRESH),
              "expected_per_contrast": exp_chance, "n_regions": n_reg,
              "n_null_perms": N_BSR_NULL}, index=[0]).to_csv(
    FIGURE_DATA_DIR / f"bsr_null_{RUN_TAG}.csv", index=False)

## 9. Join saliences to the per-region OLS

One table per contrast carrying estimate, SE, t, p, q, salience, unit salience,
and BSR. This is the object the manuscript figures and the supplemental data
file are built from, and it is the only place the two analyses meet.

In [ ]:
bsr_long = pd.concat([
    pd.DataFrame({"contrast": nm, "region": regions, "BSR": BSR[j],
                  "boot_se": R_boot_se[j]})
    for j, nm in enumerate(NAMES)], ignore_index=True)

region_table = region_lmm.merge(bsr_long, on=["contrast", "region"], how="left")
region_table["abs_BSR"]  = region_table.BSR.abs()
region_table["stable"]   = region_table.abs_BSR > BSR_THRESH
region_table["run_tag"]  = RUN_TAG
region_table["gain_normalized"] = GAIN_NORMALIZE

out = FIGURE_DATA_DIR / f"region_table_{RUN_TAG}.csv"
region_table.to_csv(out, index=False)
print(f"wrote {out}  ({len(region_table)} rows = {k_con} contrasts x {n_reg} regions)")

print("\nAgreement between the two region-level readouts:")
for nm in NAMES:
    s = region_table[region_table.contrast == nm]
    both = int((s.sig_q20 & s.stable).sum())
    print(f"  {nm:16s} q<{FDR_Q}: {int(s.sig_q20.sum()):4d}   "
          f"|BSR|>{BSR_THRESH}: {int(s.stable.sum()):4d}   both: {both:4d}")
print("\nThese are the same estimate under two error models, so overlap is")
print("expected and is NOT independent confirmation.")

---

# 10. Figure 5 panels

Each panel writes a tidy CSV alongside it so any panel can be rebuilt in Prism
or Illustrator without rerunning. The matplotlib output is for checking numbers,
not for the manuscript.

**Which matrix each panel uses.** 10.1-10.3 use `Xl` -- raw log density, no
residualization, no centering -- and this is not negotiable: the intercept of
the dark/light fit *is* the amplitude estimate, so centering would set the
measured quantity to zero by construction. 10.4 onward uses `Xz`.

In [ ]:
_MANIFEST = []

def save_panel(fig, name):
    for ext in ("pdf", "png"):
        p = FIGURE_DIR / f"{name}_{RUN_TAG}.{ext}"
        fig.savefig(p)
        _MANIFEST.append(("figure", str(p)))
    print(f"  figure -> {FIGURE_DIR / (name + '_' + RUN_TAG + '.pdf')}")

def save_data(df, name, note=""):
    p = FIGURE_DATA_DIR / f"{name}_{RUN_TAG}.csv"
    df.to_csv(p, index=False)
    _MANIFEST.append(("data", str(p)))
    print(f"  data   -> {p}  ({len(df)} rows){('  # ' + note) if note else ''}")

print(f"figures     -> {FIGURE_DIR.resolve()}")
print(f"figure data -> {FIGURE_DATA_DIR.resolve()}")

## 10.1 Panel 5A -- regional profile, dark vs light by group

One point per region per group: group mean log density in dark on x, in light on
y. The regression is fit **in log space**, which is what makes the slope
interpretable and matches the additive-in-logs scalar model.

| result | interpretation |
|---|---|
| slope = 1, intercept = 0 | no effect; points on identity |
| slope = 1, intercept > 0 | uniform multiplicative scaling; pattern preserved |
| slope < 1 | scaling that is **not** uniform -- high-baseline regions increase proportionally less |
| poor fit | regional recruitment; regions depart from any single line |

### FLAG ### The manuscript currently asserts "slope below one" and "positive
intercept" with no statistics attached. This cell supplies them: slope,
intercept, CIs, and explicit tests against slope = 1 and intercept = 0.

### CHOICE ### CIs are bootstrapped **over animals**, not over regions. Regions
are spatially autocorrelated and hierarchically nested, so region-level OLS
intervals are anticonservative by an unknown factor. Resampling animals within
cell respects the actual unit of independence.

In [ ]:
LOG_FLOOR = np.log(PSEUDO)
order = [("cre/+", "M"), ("cre/+", "F"), ("cre/cre", "M"), ("cre/cre", "F")]
N_BOOT_FIT = 2_000

# ### CHOICE ### INTERSECT_MASKS
#   True  -> all four groups fit the SAME regions (union of floored/missing across
#            groups is dropped everywhere). Slopes and intercepts are then directly
#            comparable, which is what Figure 5B asks the reader to do.
#   False -> each group keeps every region it can support. More data per fit, but
#            the four fits are over different region sets and differences between
#            them partly reflect which regions each one happened to include.
# Comparability is the point of the panel, so True is the default.
INTERSECT_MASKS = True

ci = lambda a: (np.percentile(a, 2.5), np.percentile(a, 97.5))

def group_region_means(g, sx, l, mat=Xl):
    """Per-region mean log density for one genotype x sex x light group."""
    m = ((info.genotype == g) & (info.sex == sx) & (info.light == l)).values
    with np.errstate(invalid="ignore"):
        return np.nanmean(mat[m], axis=0)

def fit_logspace(x, y):
    A = np.vstack([np.ones_like(x), x]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    b0, b1 = coef
    resid = y - (b0 + b1 * x)
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - (resid ** 2).sum() / ss_tot if ss_tot > 0 else np.nan
    return b0, b1, resid.std(ddof=2), r2

# ---- PASS 1: group means and each group's unusable regions ------------------
group_means, on_floor_by_group = {}, {}
for g, sx in order:
    xd, yl = group_region_means(g, sx, "D"), group_region_means(g, sx, "L")
    group_means[(g, sx)] = (xd, yl)
    on_floor_by_group[(g, sx)] = (np.isclose(xd, LOG_FLOOR) | np.isclose(yl, LOG_FLOOR)
                                  | ~np.isfinite(xd) | ~np.isfinite(yl))

any_floored = np.any(np.vstack([on_floor_by_group[k] for k in order]), axis=0)
keep_common = ~any_floored

print("Regions unusable per group (floored at the pseudocount or absent):")
for g, sx in order:
    own = int(on_floor_by_group[(g, sx)].sum())
    print(f"  {GROUP_LABEL[(g, sx)]:12s} {own:4d} of {n_reg}")
print(f"  union across groups  {int(any_floored.sum()):4d}"
      f"  -> common set = {int(keep_common.sum())} regions")
if INTERSECT_MASKS:
    extra = int(any_floored.sum()) - max(int(on_floor_by_group[k].sum()) for k in order)
    print(f"  intersecting costs {extra} regions beyond the worst single group")

# ---- PASS 2: fit each group on the common set ------------------------------
profile_rows, fit_rows = [], []
for g, sx in order:
    xd, yl = group_means[(g, sx)]
    on_floor = on_floor_by_group[(g, sx)]
    keep_fixed = keep_common if INTERSECT_MASKS else ~on_floor

    b0, b1, rsd, r2 = fit_logspace(xd[keep_fixed], yl[keep_fixed])

    # ### CHOICE ### Bootstrap over ANIMALS within design cell, not over regions.
    # The fit has ~270 points but only 3-8 mice per cell; regions are spatially
    # autocorrelated and hierarchically nested, so region-level OLS intervals
    # would be far too narrow. Animals are the unit of independence.
    # The region set is held fixed across replicates: recomputing it per replicate
    # would let each one fit different regions, inflating the CI and letting the
    # bootstrap estimand drift from the point estimate it should bracket.
    boots = np.empty((N_BOOT_FIT, 4))
    pos_d = np.where(((info.genotype == g) & (info.sex == sx) & (info.light == "D")).values)[0]
    pos_l = np.where(((info.genotype == g) & (info.sex == sx) & (info.light == "L")).values)[0]
    for b in range(N_BOOT_FIT):
        with np.errstate(invalid="ignore"):
            xb = np.nanmean(Xl[rng.choice(pos_d, len(pos_d), replace=True)], axis=0)
            yb = np.nanmean(Xl[rng.choice(pos_l, len(pos_l), replace=True)], axis=0)
        if not (np.isfinite(xb[keep_fixed]).all() and np.isfinite(yb[keep_fixed]).all()):
            boots[b, :] = np.nan          # resample lost a region entirely
            continue
        boots[b, :] = fit_logspace(xb[keep_fixed], yb[keep_fixed])

    boots = boots[np.isfinite(boots).all(axis=1)]
    if len(boots) < 0.9 * N_BOOT_FIT:
        print(f"  !! {GROUP_LABEL[(g, sx)]}: only {len(boots)}/{N_BOOT_FIT} bootstrap "
              f"replicates usable (a resample dropped every animal for some region)")
    b0_lo,  b0_hi  = ci(boots[:, 0])
    b1_lo,  b1_hi  = ci(boots[:, 1])
    rsd_lo, rsd_hi = ci(boots[:, 2])
    r2_lo,  r2_hi  = ci(boots[:, 3])

    # Two-sided bootstrap p by inversion. Floored at 1/N_BOOT_FIT -- report any
    # value at the floor as p < 0.0005, never as a point estimate.
    p_slope_1 = 2 * min((boots[:, 1] >= 1).mean(), (boots[:, 1] <= 1).mean())
    p_int_0   = 2 * min((boots[:, 0] >= 0).mean(), (boots[:, 0] <= 0).mean())

    fit_rows.append({"panel": "profile_fits", "group": GROUP_LABEL[(g, sx)],
                     "genotype": g, "sex": sx,
                     "n_mice_dark": len(pos_d), "n_mice_light": len(pos_l),
                     "n_regions": int(keep_fixed.sum()),
                     "n_floored_this_group": int(on_floor.sum()),
                     "n_dropped_by_intersection": int((keep_common ^ ~on_floor).sum())
                                                  if INTERSECT_MASKS else 0,
                     "intersected": INTERSECT_MASKS,
                     "intercept": b0, "intercept_lo": b0_lo, "intercept_hi": b0_hi,
                     "p_intercept_vs_0": max(p_int_0, 1 / N_BOOT_FIT),
                     "slope": b1, "slope_lo": b1_lo, "slope_hi": b1_hi,
                     "p_slope_vs_1": max(p_slope_1, 1 / N_BOOT_FIT),
                     "resid_sd": rsd, "resid_sd_lo": rsd_lo, "resid_sd_hi": rsd_hi,
                     "r2": r2, "r2_lo": r2_lo, "r2_hi": r2_hi})

    profile_rows.append(pd.DataFrame({
        "panel": "regional_profile", "group": GROUP_LABEL[(g, sx)],
        "genotype": g, "sex": sx, "region": regions, "division": region_division,
        "log_dark": xd, "log_light": yl,
        "on_floor": on_floor,              # unusable in THIS group
        "in_fit": keep_fixed}))            # actually entered the fit

profile_df = pd.concat(profile_rows, ignore_index=True)
fits_df    = pd.DataFrame(fit_rows)
save_data(profile_df, "panel5A_regional_profile")
save_data(fits_df,    "panel5A_fits", "slope/intercept/fit + bootstrap CIs and tests")

print(fits_df[["group", "n_mice_dark", "n_mice_light", "n_regions",
               "n_floored_this_group", "slope", "slope_lo", "slope_hi", "p_slope_vs_1",
               "intercept", "intercept_lo", "intercept_hi", "p_intercept_vs_0",
               "resid_sd", "r2"]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# --- checks ----------------------------------------------------------------
assert not INTERSECT_MASKS or fits_df.n_regions.nunique() == 1, \
    "intersected fits must all use the same number of regions"
if not INTERSECT_MASKS and fits_df.n_regions.nunique() > 1:
    print(f"\n!! groups fit different region sets: {fits_df.n_regions.tolist()}")
    print("   Slopes are not strictly comparable. Set INTERSECT_MASKS = True.")
if (fits_df[["n_mice_dark", "n_mice_light"]].min().min()) < 4:
    print(f"\n!! smallest cell has n={fits_df[['n_mice_dark','n_mice_light']].min().min()}; "
          "expect visibly wider CIs for that group -- that is honest, not a bug.")
at_floor = fits_df[(fits_df.p_slope_vs_1 <= 1.5 / N_BOOT_FIT) |
                   (fits_df.p_intercept_vs_0 <= 1.5 / N_BOOT_FIT)]
if len(at_floor):
    print(f"\n!! bootstrap p at the {1/N_BOOT_FIT:.4f} floor for: "
          f"{at_floor.group.tolist()} -- report as p < {1/N_BOOT_FIT:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), sharex=True, sharey=True)
for ax, (g, sx) in zip(axes, order):
    sub = profile_df[(profile_df.genotype == g) & (profile_df.sex == sx) & ~profile_df.on_floor]
    f   = fits_df[(fits_df.genotype == g) & (fits_df.sex == sx)].iloc[0]
    ax.scatter(sub.log_dark, sub.log_light, s=6, alpha=.45,
               color=GROUP_COLORS[(g, sx)], edgecolor="none")
    lim = [min(sub.log_dark.min(), sub.log_light.min()),
           max(sub.log_dark.max(), sub.log_light.max())]
    ax.plot(lim, lim, ls="--", lw=.8, color="gray", zorder=0)
    xs = np.linspace(*lim, 50)
    ax.plot(xs, f.intercept + f.slope * xs, lw=1.4, color="black")
    ax.set_title(f"{GROUP_LABEL[(g, sx)]}\nslope {f.slope:.2f} "
                 f"[{f.slope_lo:.2f},{f.slope_hi:.2f}]  int {f.intercept:+.2f}", fontsize=7)
    ax.set_xlabel("log density, dark")
axes[0].set_ylabel("log density, light")
fig.tight_layout(); save_panel(fig, "panel5A_regional_profile"); plt.show()

## 10.2 Panel 5B -- slope, intercept, and fit summary

Your comment on the draft figure. Three small panels turning the 5A scatter into
reportable numbers, with the reference lines (slope = 1, intercept = 0) drawn so
the comparison is visual as well as tabulated.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.5, 2.9))
x = np.arange(len(fits_df))
cols = [GROUP_COLORS[(r.genotype, r.sex)] for _, r in fits_df.iterrows()]

for ax, (col, lo, hi, ref, lab) in zip(axes, [
        ("slope", "slope_lo", "slope_hi", 1.0, "slope"),
        ("intercept", "intercept_lo", "intercept_hi", 0.0, "intercept (log units)"),
        ("r2", None, None, None, "$R^2$")]):
    if lo:
        ax.errorbar(x, fits_df[col],
                    yerr=[fits_df[col] - fits_df[lo], fits_df[hi] - fits_df[col]],
                    fmt="o", ms=5, capsize=3, lw=1, color="black", zorder=3)
    ax.scatter(x, fits_df[col], s=45, c=cols, zorder=4, edgecolor="black", lw=.5)
    if ref is not None:
        ax.axhline(ref, ls="--", lw=.8, color="gray")
    ax.set_xticks(x); ax.set_xticklabels(fits_df.group, rotation=40, ha="right", fontsize=7)
    ax.set_ylabel(lab)
fig.tight_layout(); save_panel(fig, "panel5B_fit_summary"); plt.show()

### 10.2a Is compression uniform across divisions?

The manuscript claims the slope effect is not a counting artifact because slopes
do not differ across major divisions. That is an accepted null doing real work,
so it needs a number rather than an assertion.

### FLAG ### Purely descriptive. Divisions differ enormously in region count,
regions within a division are spatially autocorrelated and hierarchically
nested, and there is no valid null here without the spatial-null machinery.
Report the spread of slopes, not a test.

In [ ]:
MIN_REGIONS = 8
div_rows = []
for g, sx in order:
    sub = profile_df[(profile_df.genotype == g) & (profile_df.sex == sx) & ~profile_df.on_floor]
    for d in sub.division.unique():
        dd = sub[sub.division == d]
        if len(dd) < MIN_REGIONS:
            continue
        b0, b1, rsd, r2 = fit_logspace(dd.log_dark.values, dd.log_light.values)
        div_rows.append({"panel": "division_fits", "group": GROUP_LABEL[(g, sx)],
                         "genotype": g, "sex": sx, "division": d, "n_regions": len(dd),
                         "intercept": b0, "slope": b1, "resid_sd": rsd, "r2": r2})
div_fits = pd.DataFrame(div_rows)
save_data(div_fits, "panel5B_division_fits", f"per-division fits, min {MIN_REGIONS} regions")

spread = (div_fits.groupby("group")["slope"]
          .agg(["count", "mean", "std", "min", "max"]))
spread["range"] = spread["max"] - spread["min"]
print("Per-division slope spread within each group:")
print(spread.to_string(float_format=lambda v: f"{v:.3f}"))
print("\nCompare 'range' against the whole-brain slope CIs in fits_df. If the")
print("division slopes sit inside those CIs, the compression is not division-specific.")

## 10.3 Panel 5A residuals -- what the line does not explain

Residual from each group's fitted line against dark baseline. A featureless band
means the group's light effect is fully described by slope and intercept, i.e.
pure amplitude with no regional reorganization. Structure here is the
**mixture** case, and it is the visual companion to the Section 6 counts.

Plotting centered data alongside uncentered data in one panel would not be
honest: per-mouse centering forces the intercept to zero by construction, so the
two clouds would differ exactly as much as the arithmetic guarantees.

In [ ]:
resid_rows = []
for g, sx in order:
    sub = profile_df[(profile_df.genotype == g) & (profile_df.sex == sx)].copy()
    f = fits_df[(fits_df.genotype == g) & (fits_df.sex == sx)].iloc[0]
    sub["fitted"]   = f.intercept + f.slope * sub.log_dark
    sub["residual"] = sub.log_light - sub.fitted
    resid_rows.append(sub)
resid_df = pd.concat(resid_rows, ignore_index=True)
resid_df["panel"] = "profile_residuals"
save_data(resid_df[["panel", "group", "genotype", "sex", "region", "division",
                    "log_dark", "log_light", "fitted", "residual", "on_floor"]],
          "panel5A_residuals")

fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for ax, (g, sx) in zip(axes, order):
    sub = resid_df[(resid_df.genotype == g) & (resid_df.sex == sx) & ~resid_df.on_floor]
    ax.scatter(sub.log_dark, sub.residual, s=6, alpha=.45,
               color=GROUP_COLORS[(g, sx)], edgecolor="none")
    ax.axhline(0, color="gray", lw=.8, ls="--")
    ax.set_title(f"{GROUP_LABEL[(g, sx)]}  (resid SD {sub.residual.std():.2f})", fontsize=7)
    ax.set_xlabel("log density, dark")
axes[0].set_ylabel("residual")
fig.tight_layout(); save_panel(fig, "panel5A_residuals"); plt.show()

## 10.4 Panel 5D -- per-contrast magnitudes

The panel carrying the multivariate inference. Seven bars, one per contrast,
each annotated with its **one-tailed** permutation p and the null scheme.

Bars without permutation support are drawn desaturated rather than omitted: the
flatness of the spectrum is itself informative.

### FLAG ### Sex-involving contrasts are hatched. Their p-values are against a
null that cannot fully null an interaction and must be reported as untested.

In [ ]:
cr = contrast_results.copy()
cr["supported"] = (cr["p_one_tailed"] < 0.05) & (~cr.contrast.isin(SEX_TERMS))
cr["sex_involved"] = cr.contrast.isin(SEX_TERMS)
cr = cr.sort_values("magnitude", ascending=False).reset_index(drop=True)
cr.insert(0, "panel", "contrast_magnitudes")
save_data(cr, "panel5D_contrast_magnitudes", "7 contrasts, magnitude + one-tailed p")

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.bar(range(len(cr)), cr["magnitude"], edgecolor="black", lw=.5,
       color=["#C44E52" if s else "#CCCCCC" for s in cr["supported"]],
       hatch=["//" if s else "" for s in cr["sex_involved"]])
for i, r in cr.iterrows():
    lab = f"p={r['p_one_tailed']:.3f}"
    if r["at_floor"]:
        lab = f"p<{r['floor']:.4f}"
    if not np.isnan(r["p_litter"]):
        lab += f"\n({r['p_litter']:.3f} litter)"
    ax.text(i, r["magnitude"], lab, ha="center", va="bottom", fontsize=6)
ax.set_xticks(range(len(cr)))
ax.set_xticklabels(cr["contrast"], rotation=40, ha="right", fontsize=7)
ax.set_ylabel("$\\|R_j\\|$  (multivariate magnitude)")
ax.set_title("hatched = sex-involving, null cannot fully null an interaction", fontsize=7)
fig.tight_layout(); save_panel(fig, "panel5D_contrast_magnitudes"); plt.show()

## 10.5 Panel 5E -- volcano, per-region OLS on normalized data

Estimate against uncorrected `-log10(p)`, one panel per contrast that cleared
Section 7, with the FDR thresholds drawn as horizontal lines so the reader can
map the axis onto the q-based claims.

### CHOICE ### The q < 0.2 and q < 0.05 lines are drawn at the largest p that
achieves each q, which is exact for Benjamini-Hochberg. Drawing a nominal 0.05
line instead would be misleading.

In [ ]:
SHOW = [nm for nm in NAMES if nm in ("light", "geno:sex:light", "genotype")]

fig, axes = plt.subplots(1, len(SHOW), figsize=(4.0 * len(SHOW), 3.4), sharey=True)
for ax, nm in zip(np.atleast_1d(axes), SHOW):
    s = region_table[region_table.contrast == nm]
    ax.scatter(s.estimate, s.neglog10_p, s=7, alpha=.5, edgecolor="none",
               c=np.where(s.sig_q05, "#C44E52", np.where(s.sig_q20, "#E0A458", "#BBBBBB")))
    for q, ls in [(FDR_Q, ":"), (FDR_Q_STRICT, "--")]:
        ok = s.p[s.q < q]
        if len(ok):
            ax.axhline(-np.log10(ok.max()), ls=ls, lw=.8, color="gray")
            ax.text(ax.get_xlim()[1], -np.log10(ok.max()), f" q<{q}",
                    fontsize=6, va="bottom", ha="right", color="gray")
    ax.axvline(0, color="gray", lw=.5)
    ax.set_title(f"{nm}  ({int(s.sig_q20.sum())} at q<{FDR_Q})", fontsize=8)
    ax.set_xlabel("estimate (normalized units)")
np.atleast_1d(axes)[0].set_ylabel("$-\\log_{10}$(uncorrected p)")
fig.tight_layout(); save_panel(fig, "panel5E_volcano_normalized"); plt.show()

# ### FIXED ### was `region_table[["panel_dummy"]].assign(**{}) if False else ...`
save_data(region_table.assign(panel="volcano_normalized"),
          "panel5E_region_table", "full per-region table, all contrasts")

## 10.6 Panel 5F -- significant regions per contrast

Counts at both FDR thresholds, with the `|BSR|` chance expectation shown for
reference. If the counts are small relative to the uncentered per-region models,
that is the finding.

In [ ]:
cnt = (region_table.groupby("contrast")
       .agg(q20=("sig_q20", "sum"), q05=("sig_q05", "sum"),
            stable=("stable", "sum"), n_tested=("region", "size"))
       .reindex(NAMES).reset_index())
cnt.insert(0, "panel", "sig_region_counts")
cnt["bsr_chance_expected"] = exp_chance
save_data(cnt, "panel5F_sig_region_counts")

fig, ax = plt.subplots(figsize=(5.6, 3.2))
w = 0.38
xs = np.arange(len(cnt))
ax.bar(xs - w/2, cnt.q20, w, label=f"q < {FDR_Q}", color="#E0A458", edgecolor="black", lw=.5)
ax.bar(xs + w/2, cnt.q05, w, label=f"q < {FDR_Q_STRICT}", color="#C44E52", edgecolor="black", lw=.5)
ax.axhline(exp_chance, ls=":", lw=.9, color="gray")
ax.text(len(cnt) - .5, exp_chance, " |BSR| chance", fontsize=6, va="bottom", ha="right", color="gray")
ax.set_xticks(xs); ax.set_xticklabels(cnt.contrast, rotation=40, ha="right", fontsize=7)
ax.set_ylabel(f"regions (of {n_reg})"); ax.legend(fontsize=7, frameon=False)
fig.tight_layout(); save_panel(fig, "panel5F_sig_region_counts"); plt.show()

print(cnt.to_string(index=False))

## 10.7 Panels 5H-I -- atlas heatmap exports

Region-level values for the light main effect and the three-way, written in the
shape the atlas rendering code expects. Both the estimate and the BSR are
exported so either can be mapped; the manuscript should state which is shown.

In [ ]:
for nm, tag in [("light", "5H_light"), ("geno:sex:light", "5I_three_way")]:
    s = region_table[region_table.contrast == nm][
        ["region", "division", "estimate", "se", "t", "p", "q",
         "salience", "unit_salience", "BSR", "sig_q20", "sig_q05", "stable"]].copy()
    s.insert(0, "panel", f"heatmap_{tag}")
    s.insert(1, "contrast", nm)
    save_data(s, f"panel{tag}_heatmap", f"{nm}: per-region values for atlas rendering")

---

# 11. SUPPLEMENT -- mean-centered task PLS (the BraiAn variant)

Conventional task PLS as used by the BraiAn pipeline (Chiaruttini et al. 2025,
Cell Rep 44:115876, which performs "data aggregation, normalisation and
comprehensive group statistics by partial least square analysis"). Cell means,
grand mean removed, SVD, latent variables.

**Included for comparison, not for inference.** Run on the same `Xz` as the main
path so any difference is attributable to the method rather than the input.

### FLAG ### Why the main text does not use this. The SVD rotates: each latent
variable is a linear combination of contrasts, so no LV maps onto a single
design contrast and none can be attributed to, say, the three-way interaction.
Row-normalizing `R` instead preserves per-contrast identity. That is the
methodological argument, and it belongs in Methods.

In [ ]:
def task_pls(X, cidx):
    """Mean-centered task PLS. M = cell means, grand mean removed, then SVD."""
    M  = cell_means(X, cidx)
    Mc = M - M.mean(axis=0, keepdims=True)     # remove grand mean across cells
    U, s, Vt = svd(Mc, full_matrices=False)
    return U, s, Vt.T, Mc

# ### CHANGED ### The SVD supplement runs on NULLSET, not the ANALYSIS set.
# Singular values scale with the number of regions, so the observed spectrum and
# its permutation null (11.1) must come from the same set -- the same
# norm-comparability constraint as Section 7.
Xz_lv = Xz[:, NULL_POS]
regions_lv = [regions[i] for i in NULL_POS]
division_lv = region_division[NULL_POS]
n_reg_lv = len(NULL_POS)
print(f"Supplement runs on NULLSET: {n_reg_lv} regions")

U_lv, sv_lv, V_lv, Mc_obs = task_pls(Xz_lv, cell_idx)
n_lv     = int(np.sum(sv_lv > 1e-10))
cov_expl = sv_lv ** 2 / (sv_lv ** 2).sum()

print(f"{n_lv} non-trivial latent variables")
for i in range(n_lv):
    print(f"  LV{i+1}: singular value {sv_lv[i]:8.2f}   covariance explained {100*cov_expl[i]:5.1f}%")
print(f"\ncumulative through LV3: {100*cov_expl[:3].sum():.1f}%")

## 11.1 LV permutation null

Shuffles animal labels and recomputes the singular values. Same caveat as
Section 7: any LV whose design salience loads on a sex-involving contrast
inherits the population-level null and is untested.

In [ ]:
def perm_lv(n_perm=N_PERM, rng=rng):
    """### FIXED ### The old version drew `idx`, then in the `not HAS_PERF` branch
    threw it away and permuted a SECOND time inside the call -- so the guard on
    `cidx_p` was checking a different shuffle from the one actually used."""
    null = np.full((n_perm, len(sv_lv)), np.nan)
    for i in range(n_perm):
        idx = (perm_rows_within_perfusion(info, rng) if HAS_PERF
               else rng.permutation(n_mice))
        cidx_p = cell_idx[idx] if HAS_PERF else cell_idx
        if len(np.unique(cidx_p)) < len(CELLS_DESIGN):
            continue
        sv = task_pls(Xz_lv[idx], cidx_p)[1]
        if np.isfinite(sv).all():
            null[i] = sv
    return null

null_lv = perm_lv()
p_lv = np.array([one_tailed_p(null_lv[:, i], sv_lv[i])[0] for i in range(len(sv_lv))])

scree = pd.DataFrame({
    "panel": "scree", "LV": np.arange(1, len(sv_lv) + 1),
    "singular_value": sv_lv, "cov_explained": cov_expl, "perm_p": p_lv,
    "dominant_contrast": [NAMES[int(np.argmax(np.abs(C.T @ U_lv[:, i])))]
                          for i in range(len(sv_lv))]})
save_data(scree, "figS_scree", "all LVs, nothing suppressed")
print(scree.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\n'dominant_contrast' names the largest element of a direction in 7-d contrast")
print("space. It is a reading aid, NOT an attribution -- the LV is a mixture.")

## 11.2 Split-half reliability

Split by whole perfusion batch rather than by mouse, since genotype and light
were randomized within batch and splitting on mice would let that block
structure leak across halves and inflate the agreement.

Two metrics, because they answer different questions:

- **whole-vector r** is diluted by the many near-zero loadings whose sign is
  essentially random across halves.
- **top-k Jaccard** measures what a claim would rest on: do the same regions
  come out on top? The chance expectation is printed alongside, because a
  Jaccard of 0.05 sounds low until you know chance is 0.048.

In [ ]:
def procrustes(V_ref, V_boot):
    """Align V_boot to V_ref. SVD sign and column order are arbitrary across
    resamples, so rotate before comparing.

    ### FIXED ### Two shape traps, both live here:
      1. Mean-centering an 8 x p matrix drops its rank to 7, so `task_pls`
         returns V as p x 8 while n_lv is 7. Slice both inputs to a common
         column count or the product is non-square.
      2. `svd(M)` defaults to full_matrices=True, which on a non-square M
         returns Uu and Vvt with incompatible inner dimensions. Must be False.
    """
    L = min(V_ref.shape[1], V_boot.shape[1])
    V_ref, V_boot = V_ref[:, :L], V_boot[:, :L]
    Uu, _, Vvt = svd(V_boot.T @ V_ref, full_matrices=False)
    return V_boot @ (Uu @ Vvt)

def split_half_all(n_splits=N_SPLITS, k=TOPK, rng=rng):
    if not HAS_PERF:
        print("!! no perfusion_batch column -- skipping")
        return None
    batches = info["perfusion_batch"].unique()
    whole, jac = [], []
    for _ in range(n_splits):
        h1 = rng.choice(batches, size=len(batches) // 2, replace=False)
        m1 = info["perfusion_batch"].isin(h1).values
        if len(np.unique(cell_idx[m1])) < 8 or len(np.unique(cell_idx[~m1])) < 8:
            continue
        _, _, V1, _ = task_pls(Xz_lv[m1],  cell_idx[m1])
        _, _, V2, _ = task_pls(Xz_lv[~m1], cell_idx[~m1])
        if not (np.isfinite(V1).all() and np.isfinite(V2).all()):
            continue          # a half emptied a cell for some region
        V2 = procrustes(V1, V2)
        w_row, j_row = [], []
        for LV in range(n_lv):
            w_row.append(abs(pearsonr(V1[:, LV], V2[:, LV])[0]))
            t1 = set(np.argsort(-np.abs(V1[:, LV]))[:k])
            t2 = set(np.argsort(-np.abs(V2[:, LV]))[:k])
            j_row.append(len(t1 & t2) / len(t1 | t2))
        whole.append(w_row); jac.append(j_row)
    return {"whole_r": np.array(whole), "jaccard": np.array(jac)}

sh = split_half_all()
chance_jac = TOPK / (2 * n_reg_lv - TOPK)
print(f"chance Jaccard for top-{TOPK} of {n_reg_lv} regions = {chance_jac:.4f}\n")
if sh is not None:
    sh_df = pd.DataFrame({
        "panel": "split_half", "LV": np.arange(1, n_lv + 1),
        "whole_r_mean": sh["whole_r"].mean(0),
        "jaccard_mean": sh["jaccard"].mean(0),
        "jaccard_lo": np.percentile(sh["jaccard"], 2.5, axis=0),
        "jaccard_hi": np.percentile(sh["jaccard"], 97.5, axis=0),
        "chance_jaccard": chance_jac, "n_splits": len(sh["whole_r"])})
    sh_df["above_chance"] = sh_df.jaccard_lo > chance_jac
    save_data(sh_df, "figS_split_half")
    print(sh_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    if not sh_df.above_chance.any():
        print("\n-> No LV exceeds chance overlap. The LVs are exploratory descriptions")
        print("   of covariance structure in this dataset and nothing more. Any claim")
        print("   about an LV must rest on external anatomy, not on reproducibility.")

## 11.3 LV design-cell projection and scree panel

`C @ U` maps each LV's design salience back onto the eight design cells --
the form the PLS literature expects, and the only readable way to see what a
component separates.

### FLAG ### Close to tautological as evidence: `C @ U` is a deterministic
function of the design coding and the salience vector, so it will always look
structured. A reading aid, not a result.

In [ ]:
# ### FIXED ### U_lv comes from the SVD of the 8 x p cell-mean matrix, so it is
# ALREADY in design-cell space (8 x L). The old SVD-of-R version produced U in
# contrast space (7 x L) and needed C @ U to get here; that step is wrong now.
assert U_lv.shape[0] == len(CELLS_DESIGN), \
    f"U_lv should be {len(CELLS_DESIGN)} x L (cell space), got {U_lv.shape}"

proj = U_lv                                    # 8 x L, design-cell saliences

proj_df = pd.DataFrame([
    {"panel": "lv_cell_projection", "LV": LV + 1, "genotype": g, "sex": sx, "light": l,
     "group": GROUP_LABEL[(g, sx)], "n": int((cell_idx == i).sum()),
     "design_score": proj[i, LV]}
    for i, (g, sx, l) in enumerate(CELLS_DESIGN) for LV in range(n_lv)])
save_data(proj_df, "figS_lv_cell_projection", "U_lv, 8 cells x n_lv")

# Contrast-space coordinates of each LV: C' U tells you which design contrasts
# the component loads on. This is the direction the OLD code was trying to go,
# and it is the one that supports the "LVs are mixtures" argument.
contrast_coord = C.T @ U_lv[:, :n_lv]          # 7 x n_lv
coord_df = pd.DataFrame(contrast_coord, index=NAMES,
                        columns=[f"LV{i+1}" for i in range(n_lv)])
coord_df = (coord_df / np.linalg.norm(coord_df.values, axis=0, keepdims=True)).round(3)
print("LV loadings in contrast space (columns are unit-normalized):")
print(coord_df.to_string())
print("\nA column with several comparable entries is a mixture -- it cannot be")
print("attributed to any single design contrast. That is the argument for the")
print("unrotated main path.")
coord_df.reset_index().rename(columns={"index": "contrast"}).assign(
    panel="lv_contrast_loadings").to_csv(
    FIGURE_DATA_DIR / f"figS_lv_contrast_loadings_{RUN_TAG}.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.1))
axes[0].bar(np.arange(1, len(sv_lv) + 1), 100 * cov_expl, color="#6C8EBF",
            edgecolor="black", lw=.5)
for i, p_ in enumerate(p_lv):
    axes[0].text(i + 1, 100 * cov_expl[i], f"p={p_:.3f}", ha="center",
                 va="bottom", fontsize=6)
axes[0].set_xlabel("latent variable"); axes[0].set_ylabel("% covariance explained")

n_show = min(3, n_lv)
for LV in range(n_show):
    sub = proj_df[proj_df.LV == LV + 1]
    axes[1].plot(range(len(sub)), sub.design_score, marker="o", ms=4,
                 label=f"LV{LV+1}", lw=1)
axes[1].set_xticks(range(len(CELLS_DESIGN)))
axes[1].set_xticklabels([f"{GROUP_LABEL[(g,s)]} {l}" for g, s, l in CELLS_DESIGN],
                        rotation=55, ha="right", fontsize=6)
axes[1].axhline(0, color="gray", lw=.5)
axes[1].set_ylabel("design salience"); axes[1].legend(fontsize=7, frameon=False)
fig.tight_layout(); save_panel(fig, "figS_scree_and_projection"); plt.show()

## 11.4 LV region saliences with BSR

Bootstrap SEs for `V`, Procrustes-aligned because SVD sign and column order are
arbitrary across resamples. Same uncorrected caveat as Section 8.

In [ ]:
def bootstrap_V(X, V_ref, n_boot, rng):
    p, L = V_ref.shape
    s1 = np.zeros((p, L)); s2 = np.zeros((p, L))
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, size=len(pos), replace=True)
                              for pos in CELL_POS])
        _, _, Vb, _ = task_pls(X[idx], cell_idx[idx])
        Vb = procrustes(V_ref, Vb[:, :L])      # trim rank-deficient extra column
        s1 += Vb; s2 += Vb ** 2
    m   = s1 / n_boot
    var = np.maximum(s2 / n_boot - m ** 2, 0) * n_boot / (n_boot - 1)
    return m, np.sqrt(var)

V_ref = V_lv[:, :n_lv]
assert V_ref.shape == (n_reg_lv, n_lv), f"V_ref should be {n_reg_lv} x {n_lv}, got {V_ref.shape}"

V_bm, V_bse = bootstrap_V(Xz_lv, V_ref, N_BOOT, rng)
BSR_V = np.divide(V_bm, V_bse, out=np.zeros_like(V_bm), where=V_bse > 0)

lv_long = pd.concat([
    pd.DataFrame({"panel": "lv_region_saliences", "LV": LV + 1,
                  "region": regions_lv, "division": division_lv,
                  "salience": V_lv[:, LV], "BSR": BSR_V[:, LV],
                  "abs_BSR": np.abs(BSR_V[:, LV])})
    for LV in range(n_lv)], ignore_index=True)
save_data(lv_long, "figS_lv_region_saliences", "long format, all LVs")

print(f"Stable regions per LV (|BSR| > {BSR_THRESH}):")
print(f"  empirical chance from Section 8: {exp_chance:.0f} of {n_reg_lv} per component")
for LV in range(n_lv):
    obs = int((np.abs(BSR_V[:, LV]) > BSR_THRESH).sum())
    print(f"  LV{LV+1}: {obs:4d}")
print("\nSame small-n SE bias applies here as in Section 8 -- do not quote the")
print("normal-theory 5%. Descriptive only; BSR is not used for significance.")

## 11.5 Main path vs. supplement

The comparison a reviewer will want: do the unrotated per-contrast saliences and
the rotated LV saliences describe the same regions? A high correlation means the
two methods agree and the choice is presentational. A low one means the rotation
genuinely mixed things, which is the argument for the main path.

In [ ]:
cmp_rows = []
for j, nm in enumerate(NAMES):
    for LV in range(n_lv):
        # No p-value: regions are spatially autocorrelated and hierarchically
        # nested, so the nominal df here is meaningless. The magnitude is the
        # descriptive quantity; it is not being tested.
        r, _ = pearsonr(R_unit[j][NULL_POS], V_lv[:, LV])
        cmp_rows.append({"contrast": nm, "LV": LV + 1, "r": r, "abs_r": abs(r)})
cmp_df = pd.DataFrame(cmp_rows)

piv = cmp_df.pivot(index="contrast", columns="LV", values="r").reindex(NAMES)
best = cmp_df.loc[cmp_df.groupby("contrast")["abs_r"].idxmax()].set_index("contrast").reindex(NAMES)
cmp_df["is_best_lv"] = False
cmp_df.loc[cmp_df.groupby("contrast")["abs_r"].idxmax(), "is_best_lv"] = True
save_data(cmp_df.assign(panel="contrast_vs_lv"), "figS_contrast_vs_lv")

# ---------------------------------------------------------------- heatmap
M = piv.values
fig, ax = plt.subplots(figsize=(1.05 * n_lv + 3.4, 0.52 * len(NAMES) + 2.0))

# Diverging map on FIXED symmetric limits. A correlation heatmap autoscaled to
# its own range makes r = 0.3 look saturated; -1..1 keeps the panel readable
# against any other correlation figure in the paper.
im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")

for i in range(M.shape[0]):
    for k in range(M.shape[1]):
        v = M[i, k]
        ax.text(k, i, f"{v:+.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > 0.55 else "black")

# Box the best-matching LV per contrast -- the row maximum is the claim being made.
#for i, nm in enumerate(NAMES):
#    k = int(best.loc[nm, "LV"]) - 1
#    ax.add_patch(plt.Rectangle((k - .5, i - .5), 1, 1, fill=False,
#                               edgecolor="black", lw=2.0, zorder=5))

ax.set_xticks(range(n_lv), [f"LV{k+1}" for k in range(n_lv)])
ax.set_yticks(range(len(NAMES)), NAMES)
ax.set_xlabel("latent variable (SVD, supplement)")
#ax.set_title("Unrotated contrast saliences vs. LV region saliences\n"
#             f"Pearson r across {len(NULL_POS)} regions; boxed = largest |r| per contrast",
#             fontsize=10)
cbar = fig.colorbar(im, ax=ax, fraction=0.030, pad=0.02)
cbar.set_label("", rotation=0, labelpad=10)
ax.set_xticks(np.arange(-.5, n_lv, 1), minor=True)
ax.set_yticks(np.arange(-.5, len(NAMES), 1), minor=True)
ax.grid(which="minor", color="white", lw=1.2)
ax.tick_params(which="minor", length=0)
plt.tight_layout()
save_panel(fig, "figS_contrast_vs_lv_heatmap")
plt.show()

print("Correlation between unit contrast saliences and LV region saliences:")
print(piv.to_string(float_format=lambda v: f"{v:+.2f}"))
print("\nBest-matching LV per contrast:")
print(best[["LV", "r", "abs_r"]].to_string(float_format=lambda v: f"{v:+.3f}"))
print(f"\nlargest |r| anywhere in the matrix: {cmp_df.abs_r.max():.3f}")
print("\nA contrast whose best |r| is well below 1 is not represented by any single")
print("LV -- which is exactly why the main text uses the unrotated form.")
print("\nSIGN CAVEAT: SVD component signs are arbitrary up to the convention applied")
print("in 12.x, so the sign of r is only interpretable relative to that convention.")
print("|r| is the quantity to read. No p-values are attached: regions are spatially")
print("autocorrelated and hierarchically nested, so the nominal df is meaningless.")

---

# 12. Manifest

Every figure and CSV written by this run, so the manuscript figures can be
traced back to a specific run tag.

In [ ]:
man = pd.DataFrame(_MANIFEST, columns=["kind", "path"]).drop_duplicates()
man.insert(0, "run_tag", RUN_TAG)
man_path = FIGURE_DATA_DIR / f"MANIFEST_{RUN_TAG}.csv"
man.to_csv(man_path, index=False)
print(f"{len(man)} artifacts written under run_tag={RUN_TAG}")
print(man.to_string(index=False))

print("\n" + "=" * 72)
print("REPORTING CHECKLIST")
print("=" * 72)
print(f"  R_MODE                  {R_MODE}")
print(f"  GAIN_NORMALIZE          {GAIN_NORMALIZE}")
print(f"  region inclusion        {REGION_INCLUSION} (>= {MIN_OBS_PER_CELL}/cell)")
print(f"  ANALYSIS set            {n_reg}   saliences, per-region OLS, BSR, exports")
print(f"  NULLSET (= CORE)        {len(NULL_POS)}   every permutation null, and the SVD supplement")
print(f"  regions gained vs listwise complete: +{n_reg - len(NULL_POS)}")
print(f"  mice                    {n_mice}  (smallest cell n = {n_cell.min()})")
print(f"  pseudocount             {PSEUDO:.6g}")
print(f"  permutations            {N_PERM}   (p floor {1/(N_PERM+1):.5f}, ONE-TAILED)")
print(f"  bootstraps              {N_BOOT}")
print(f"  per-region OLS df       min {int(np.nanmin(df_resid))} / "
      f"median {int(np.nanmedian(df_resid))} / max {int(np.nanmax(df_resid))}  "
      f"(now varies by region: n_obs - 8)")
print(f"  regions with no usable SE  {int(no_se.sum())} (< {MIN_TOTAL_N_FOR_SE} obs; p = NaN, excluded from FDR)")
print(f"  FDR thresholds          {FDR_Q} / {FDR_Q_STRICT}")
print(f"  BSR threshold           {BSR_THRESH}  (uncorrected)")
print(f"  BSR empirical chance    {exp_chance:.0f}/{n_reg} per contrast "
      f"({emp_rate/(2*norm.sf(BSR_THRESH)):.1f}x the normal-theory rate)")
print("\n  Sign convention: +1 = control, male, light.")
print("  RESOLVED: the global-model notebook has been flipped to match this, so all")
print("  three notebooks are now control-positive. Any genotype-bearing number from")
print("  a global-model run predating that change is NEGATED relative to this one.")
print("\n  Magnitudes: the tested magnitude is over NULLSET; the ANALYSIS-set")
print("  magnitude is descriptive. Never quote one with the other's p-value.")
print("\n  Sex-involving contrasts are tested against a null that cannot fully null")
print("  an interaction. Report as untested, not as significant.")